# A3: LDA Implementation from Scratch

**Answer:** This solution calculates class-wise and overall means, within-class scatter $S_W$, between-class scatter $S_B$, solves the regularized eigenvalue problem, selects the leading discriminant vector, projects samples, and plots the result. The source PDF does not provide a dataset, so the code creates a reproducible synthetic two-class dataset.

$S_W$ measures spread within classes; $S_B$ measures separation of class means. Eigenvectors of $S_W^{-1}S_B$ provide discriminant directions, ordered by eigenvalue. LDA can return at most $C-1$ components, so two classes produce one.

In [ ]:
import numpy as np

def lda_fit_transform(X, y, n_components=None, ridge=1e-6):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    classes = np.unique(y)
    if len(classes) < 2:
        raise ValueError("LDA requires at least two classes")

    overall_mean = X.mean(axis=0)
    n_features = X.shape[1]
    within_scatter = np.zeros((n_features, n_features))
    between_scatter = np.zeros((n_features, n_features))
    class_means = {}

    for label in classes:
        class_samples = X[y == label]
        class_mean = class_samples.mean(axis=0)
        class_means[label] = class_mean
        centered = class_samples - class_mean
        within_scatter += centered.T @ centered
        mean_delta = (class_mean - overall_mean).reshape(-1, 1)
        between_scatter += len(class_samples) * (mean_delta @ mean_delta.T)

    # Regularization stabilizes the solve when within-class scatter is singular.
    regularized_within = within_scatter + ridge * np.eye(n_features)
    matrix = np.linalg.solve(regularized_within, between_scatter)
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)
    order = np.argsort(eigenvalues)[::-1]

    max_components = min(n_features, len(classes) - 1)
    if n_components is None:
        n_components = max_components
    if not 1 <= n_components <= max_components:
        raise ValueError("n_components must be between 1 and C - 1")

    directions = eigenvectors[:, order[:n_components]]
    transformed = (X - overall_mean) @ directions
    return transformed, directions, class_means, overall_mean, within_scatter, between_scatter, eigenvalues[order]

# Reproducible synthetic example; no dataset was specified in the assignment.
rng = np.random.RandomState(7)
class_0 = rng.normal(loc=[0.0, 0.0], scale=[0.8, 0.8], size=(30, 2))
class_1 = rng.normal(loc=[2.5, 2.5], scale=[0.8, 0.8], size=(30, 2))
X = np.vstack((class_0, class_1))
y = np.array([0] * len(class_0) + [1] * len(class_1))

X_lda, W, class_means, overall_mean, S_W, S_B, eigenvalues = lda_fit_transform(X, y, n_components=1)
print("Class means:\n", class_means)
print("Overall mean:", overall_mean)
print("Within-class scatter S_W:\n", S_W)
print("Between-class scatter S_B:\n", S_B)
print("Sorted eigenvalues:", eigenvalues)
print("Selected discriminant vector:\n", W)
print("Reduced feature matrix shape:", X_lda.shape)
print("First five projected samples:\n", X_lda[:5])

try:
    import matplotlib.pyplot as plt
    plt.scatter(X_lda[y == 0, 0], np.zeros((y == 0).sum()), label="Class 0", alpha=0.75)
    plt.scatter(X_lda[y == 1, 0], np.zeros((y == 1).sum()), label="Class 1", alpha=0.75)
    plt.xlabel("LD1")
    plt.yticks([])
    plt.title("Two classes projected onto the LDA direction")
    plt.legend()
    plt.grid(True, axis="x")
    plt.show()
except ImportError:
    print("Install matplotlib to display the projected-sample plot.")